In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

import nltk
from nltk.corpus import stopwords

from transformers import BertTokenizer, BertForSequenceClassification, pipeline


In [17]:
df = pd.read_csv("rss_articles.csv")

df = df[['timestamp', 'title', 'text_body','source',]]

df['Date'] = pd.to_datetime(df['timestamp']).dt.date
df = df.dropna(subset=['text_body'])
df.shape


(432, 5)

In [4]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clear_text(x):
    words = str(x).split()
    filtered = [w for w in words if w.lower() not in stop_words]
    return " ".join(filtered)[:512]

df['text_body'] = df['text_body'].apply(clear_text)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Amogh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")
finbert = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

tqdm.pandas()
df['sent_finbert'] = df['text_body'].progress_apply(lambda x: finbert(x)[0])
df['sent_label']  = df['sent_finbert'].apply(lambda x: x['label'].lower())
df['sent_score']  = df['sent_finbert'].apply(lambda x: x['score'])

sent_map = {"neutral":0, "positive":1, "negative":-1}

df['sent_numeric'] = df['sent_label'].map(sent_map)
df['sent_weight']  = df['sent_numeric'] * df['sent_score']


Device set to use cpu
100%|██████████| 432/432 [01:36<00:00,  4.48it/s]


In [18]:
df.shape

(432, 5)

In [ ]:
daily = (
    df.groupby('Date')['sent_weight']
    .mean()
    .reset_index()
    .rename(columns={'sent_weight':'daily_sent'})
)

daily

,Date,daily_sentiment
0,2008-06-29,0.000000
1,2008-07-02,0.999997
2,2008-07-28,0.000000
3,2008-10-01,0.000000
4,2008-12-05,0.000000
5,2008-12-06,-0.999998
6,2009-10-13,0.500000
7,2010-03-05,-0.665401
8,2010-08-08,0.000000
9,2010-08-09,0.000000


In [19]:
import yfinance as yf

nifty = yf.download("^NSEI", start="2010-01-01")
nifty = nifty.reset_index()

nifty = nifty[[('Date', ''), ('Close', '^NSEI')]]
nifty.columns = ['Date', 'Close']
nifty['Date'] = nifty['Date'].dt.date
nifty.head()

C:\Users\Amogh\AppData\Local\Temp\ipykernel_17944\2005534955.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nifty = yf.download("^NSEI", start="2010-01-01")
[*********************100%***********************]  1 of 1 completed


,Date,Close
0,2010-01-04,5232.200195
1,2010-01-05,5277.899902
2,2010-01-06,5281.799805
3,2010-01-07,5263.100098
4,2010-01-08,5244.750000


In [ ]:
df_merged = pd.merge(nifty, daily, on="Date", how="left")
df_merged['daily_sent'] = df_merged['daily_sent'].fillna(0)

df_merged

,Date,Close,daily_sentiment
0,2010-01-04,5232.200195,0.0
1,2010-01-05,5277.899902,0.0
2,2010-01-06,5281.799805,0.0
3,2010-01-07,5263.100098,0.0
4,2010-01-08,5244.750000,0.0
...,...,...,...
3921,2025-12-19,25966.400391,0.0
3922,2025-12-22,26172.400391,0.0
3923,2025-12-23,26177.150391,0.0
3924,2025-12-24,26142.099609,0.0


In [9]:
df_merged['Return_1d'] = df_merged['Close'].pct_change().shift(-1)

def label_signal(r):
    if r > 0:
        return 1
    elif r < 0:
        return -1
    else:
        return 0

df_merged['signal'] = df_merged['Return_1d'].apply(label_signal)

df_merged.dropna(inplace=True)
df_merged.head()


,Date,Close,daily_sent,Return_1d,signal
0,2010-01-04,5232.200195,0.0,0.008734,1
1,2010-01-05,5277.899902,0.0,0.000739,1
2,2010-01-06,5281.799805,0.0,-0.003540,-1
3,2010-01-07,5263.100098,0.0,-0.003487,-1
4,2010-01-08,5244.750000,0.0,0.000887,1


In [ ]:
features = ['daily_sent']
X = df_merged[features]
y = df_merged['signal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


### **Logistic Regression Model**

In [ ]:
logr = LogisticRegression(max_iter=200)
logr.fit(X_train_scaled, y_train)

pred_lr = logr.predict(X_test_scaled)
print("Logistic Regression Accuracy:", accuracy_score(y_test, pred_lr))
print(classification_report(y_test, pred_lr))


Logistic Regression Accuracy: 0.5449871465295629
              precision    recall  f1-score   support

          -1       0.60      0.01      0.02       354
           0       0.00      0.00      0.00         1
           1       0.54      1.00      0.70       423

    accuracy                           0.54       778
   macro avg       0.38      0.33      0.24       778
weighted avg       0.57      0.54      0.39       778



### **Random Forest Classifier**

In [ ]:
rf = RandomForestClassifier(n_estimators=300, max_depth=5)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
print("RF Accuracy:", accuracy_score(y_test, pred_rf))
print(classification_report(y_test, pred_rf))


RF Accuracy: 0.5424164524421594
              precision    recall  f1-score   support

          -1       0.43      0.01      0.02       354
           0       0.00      0.00      0.00         1
           1       0.54      0.99      0.70       423

    accuracy                           0.54       778
   macro avg       0.32      0.33      0.24       778
weighted avg       0.49      0.54      0.39       778



### **XGBoost Classifier**

In [ ]:
label_mapping = {-1: 0, 0: 1, 1: 2}
y_train_mapped = y_train.map(label_mapping)
y_test_mapped = y_test.map(label_mapping)

xgb_clf = xgb.XGBClassifier(
    objective="multi:softmax",
    num_class=3,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05
)

xgb_clf.fit(X_train, y_train_mapped)
pred_xgb = xgb_clf.predict(X_test)

print("XGB Accuracy:", accuracy_score(y_test_mapped, pred_xgb))
print(classification_report(y_test_mapped, pred_xgb))

XGB Accuracy: 0.5437017994858612
              precision    recall  f1-score   support

           0       0.50      0.00      0.01       354
           1       0.00      0.00      0.00         1
           2       0.54      1.00      0.70       423

    accuracy                           0.54       778
   macro avg       0.35      0.33      0.24       778
weighted avg       0.52      0.54      0.39       778



### **NN - MLPClassifier**

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=400)
mlp.fit(X_train_scaled, y_train)

pred_mlp = mlp.predict(X_test_scaled)
print("MLP Accuracy:", accuracy_score(y_test, pred_mlp))
print(classification_report(y_test, pred_mlp))


MLP Accuracy: 0.5449871465295629
              precision    recall  f1-score   support

          -1       0.57      0.01      0.02       354
           0       0.00      0.00      0.00         1
           1       0.54      0.99      0.70       423

    accuracy                           0.54       778
   macro avg       0.37      0.33      0.24       778
weighted avg       0.56      0.54      0.39       778



### **Gradient Boosting Classifier**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier()
gbc.fit(X_train, y_train)

pred_gbc = gbc.predict(X_test)
print("GBC Accuracy:", accuracy_score(y_test, pred_gbc))
print(classification_report(y_test, pred_gbc))


GBC Accuracy: 0.5424164524421594
              precision    recall  f1-score   support

          -1       0.43      0.01      0.02       354
           0       0.00      0.00      0.00         1
           1       0.54      0.99      0.70       423

    accuracy                           0.54       778
   macro avg       0.32      0.33      0.24       778
weighted avg       0.49      0.54      0.39       778



### **SVC**

In [ ]:
from sklearn.svm import SVC

svm_clf = SVC(kernel="rbf", C=1.0)
svm_clf.fit(X_train_scaled, y_train)

pred_svm = svm_clf.predict(X_test_scaled)
print("SVM Accuracy:", accuracy_score(y_test, pred_svm))
print(classification_report(y_test, pred_svm))


SVM Accuracy: 0.5424164524421594
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00       354
           0       0.00      0.00      0.00         1
           1       0.54      1.00      0.70       423

    accuracy                           0.54       778
   macro avg       0.18      0.33      0.23       778
weighted avg       0.30      0.54      0.38       778

